# Enemy Agent Tournament

Runs a full round-robin benchmark across all 16 enemy agents.
Results are saved to `tournament_results.json` — use this to build the opponent curriculum for PPO training.

## Required Kaggle inputs

| Dataset slug | Contents | How to create |
|---|---|---|
| `orbitswar-enemies` | Full enemy agents dir (incl. weights) | `python zip_kaggle_dataset.py --enemies` |

## Tournament format

**1v1 Round-robin:** Every pair of agents plays once.
- 16 agents → C(16,2) = **120 games**
- Each agent plays exactly **15 games**

**4v4 Balanced:** 5 rounds × 4 groups of 4 = 20 games.
- Each agent plays exactly **5 games**
- Each pair of agents appears together **exactly once** (perfect combinatorial design)

## Runtime
- **CPU only** (no model training — GPU wastes quota)
- ~30–60s per game → **~2–3 hours total** for 140 games
- Saves after every game — safe to interrupt and rerun

## Output
- `tournament_results.json` — full results + rankings
- `tournament_curves.png` — win-rate leaderboard chart

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 1 — Install orbit_wars + verify
# ─────────────────────────────────────────────────────────
%pip install --upgrade --quiet 'kaggle-environments>=1.28.0'

from kaggle_environments import make
env = make('orbit_wars', configuration={'episodeSteps': 5})
print('orbit_wars OK')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}  (CPU recommended — no training here)')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 2 — Logging
# ─────────────────────────────────────────────────────────
import sys, time
from pathlib import Path

WORK     = Path('/kaggle/working')
LOG_PATH = WORK / 'tournament.log'

class _TeeLogger:
    def __init__(self, path):
        self._orig = sys.stdout
        self._file = open(path, 'w', buffering=1, encoding='utf-8')
        ts = time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())
        self._file.write(f'# OrbitsWar — Tournament Log\n# Started : {ts}\n\n')
    def write(self, msg):
        self._orig.write(msg)
        try: self._file.write(msg)
        except: pass
    def flush(self):
        self._orig.flush()
        try: self._file.flush()
        except: pass
    def isatty(self): return False
    def close(self):
        sys.stdout = self._orig
        try: self._file.flush(); self._file.close()
        except: pass

_tee = _TeeLogger(LOG_PATH)
sys.stdout = _tee
print(f'Logging to : {LOG_PATH}')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 3 — Dataset paths + copy enemy agents
# ─────────────────────────────────────────────────────────
import os, sys, shutil
from pathlib import Path

ENEMIES_DATASET = 'orbitswar-enemies'
ENEMIES_SRC     = Path(f'/kaggle/input/{ENEMIES_DATASET}')
ENEMIES_DST     = WORK / 'enemy agents'
RESULTS_PATH    = WORK / 'tournament_results.json'

if not ENEMIES_SRC.exists():
    raise FileNotFoundError(
        f'Dataset not found: {ENEMIES_SRC}\n'
        'Run: python zip_kaggle_dataset.py --enemies\n'
        'Upload orbitswar-enemies.zip as Kaggle dataset "orbitswar-enemies"'
    )

# Copy enemy agents to working directory (preserves relative weight paths)
enemy_src_dir = ENEMIES_SRC / 'enemy agents'
if not enemy_src_dir.exists():
    enemy_src_dir = ENEMIES_SRC   # flat layout fallback

if ENEMIES_DST.exists():
    shutil.rmtree(ENEMIES_DST)
shutil.copytree(enemy_src_dir, ENEMIES_DST)
print(f'Enemy agents copied to: {ENEMIES_DST}')

# List what we have
agent_dirs = sorted([
    p for p in ENEMIES_DST.iterdir()
    if p.is_dir() and p.name != '__pycache__'
])
print(f'Found {len(agent_dirs)} agent directories:')
for d in agent_dirs:
    has_weights = any(d.rglob('*.pt'))
    print(f'  {d.name}{"  [has weights]" if has_weights else ""}')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 4 — Load all agents
# ─────────────────────────────────────────────────────────
import importlib.util

def load_agent(agent_dir: Path):
    """
    Load main.py from an agent directory.
    Returns (name, callable) or (name, None) on failure.
    """
    name     = agent_dir.name
    main_py  = agent_dir / 'main.py'
    if not main_py.exists():
        return name, None

    # Add agent dir to sys.path so relative imports work
    agent_str = str(agent_dir)
    if agent_str not in sys.path:
        sys.path.insert(0, agent_str)

    try:
        spec = importlib.util.spec_from_file_location(f'agent_{name}', str(main_py))
        mod  = importlib.util.module_from_spec(spec)
        sys.modules[f'agent_{name}'] = mod
        spec.loader.exec_module(mod)
        fn = getattr(mod, 'agent', None)
        if fn is None:
            return name, None
        return name, fn
    except Exception as e:
        print(f'  FAILED to load {name}: {e}')
        return name, None


# Load all agents
agents = []   # list of (name, fn)
failed = []

for d in agent_dirs:
    name, fn = load_agent(d)
    if fn is not None:
        agents.append((name, fn))
        print(f'  OK  {name}')
    else:
        failed.append(name)
        print(f'  --  {name}  (skipped)')

print(f'\nLoaded: {len(agents)}  Failed: {len(failed)}')
if len(agents) < 2:
    raise RuntimeError('Need at least 2 agents to run a tournament.')

AGENT_NAMES = [name for name, _ in agents]
AGENT_FNS   = {name: fn for name, fn in agents}
N = len(agents)
print(f'Tournament size: {N} agents')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 5 — Game runner + results helpers
# ─────────────────────────────────────────────────────────
import json, time
from kaggle_environments import make

def _call_agent(fn, obs, config={}):
    """Call an agent function, handling both (obs) and (obs, config) signatures."""
    try:
        result = fn(obs, config)
    except TypeError:
        try:
            result = fn(obs)
        except Exception:
            result = []
    except Exception:
        result = []
    return result or []


def run_game(fns: list, seed: int = 42):
    """
    Run one game with len(fns) players.
    Returns list of final rewards (float), or None on crash.
    """
    n = len(fns)
    try:
        env = make('orbit_wars',
                   configuration={'seed': seed, 'randomSeed': seed},
                   debug=False)
        env.reset(num_agents=n)
        states = env.step([[]] * n)

        while True:
            # Game ends when no player is ACTIVE
            if not any(str(s.status) == 'ACTIVE' for s in states):
                break

            actions = []
            for s, fn in zip(states, fns):
                if str(s.status) == 'ACTIVE' and fn is not None:
                    obs = s.observation
                    a   = _call_agent(fn, obs)
                else:
                    a = []
                actions.append(a)

            states = env.step(actions)

        rewards = [float(s.reward or 0) for s in states]
        return rewards

    except Exception as e:
        print(f'    [game error] {e}')
        return None


def _determine_winner(rewards, names):
    """Given rewards list, return winner name or 'draw'."""
    if rewards is None:
        return 'error'
    valid = [r for r in rewards if r is not None]
    if not valid:
        return 'error'
    max_r = max(valid)
    if max_r <= 0 or valid.count(max_r) > 1:
        return 'draw'
    return names[rewards.index(max_r)]


def save_results(results, path):
    path.write_text(json.dumps(results, indent=2))


def load_results(path):
    if path.exists():
        try:
            return json.loads(path.read_text())
        except Exception:
            pass
    return None


print('Game runner ready.')
print(f'Results will be saved to: {RESULTS_PATH}')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 6 — 1v1 Round-robin tournament
#
# Every pair (i, j) where i < j plays one game.
# Total: C(N, 2) games. Each agent plays N-1 games.
# ─────────────────────────────────────────────────────────
import itertools, time

# Load or initialise results
results = load_results(RESULTS_PATH) or {
    'metadata': {
        'agents':       AGENT_NAMES,
        'n_agents':     N,
        'n_1v1_games':  N * (N - 1) // 2,
        'n_4v4_games':  0,   # filled in later
    },
    '1v1_matches': {},
    '4v4_matches': [],
}

done_1v1 = set(results['1v1_matches'].keys())
total_1v1 = N * (N - 1) // 2
remaining = total_1v1 - len(done_1v1)
print(f'1v1 Round-robin: {total_1v1} total games  ({len(done_1v1)} done, {remaining} remaining)')
print(f'Each agent plays {N-1} games.\n')

game_num = len(done_1v1)
t_start  = time.time()

for i, j in itertools.combinations(range(N), 2):
    a_name = AGENT_NAMES[i]
    b_name = AGENT_NAMES[j]
    key    = f'{a_name}__vs__{b_name}'

    if key in done_1v1:
        continue   # already played — skip

    game_num += 1
    seed = 1000 + i * 100 + j

    t0 = time.time()
    print(f'  [{game_num}/{total_1v1}] {a_name}  vs  {b_name} ... ', end='', flush=True)

    rewards = run_game([AGENT_FNS[a_name], AGENT_FNS[b_name]], seed=seed)
    winner  = _determine_winner(rewards, [a_name, b_name])
    elapsed = time.time() - t0

    print(f'{winner}  ({elapsed:.0f}s)')

    results['1v1_matches'][key] = {
        'agent_a':  a_name,
        'agent_b':  b_name,
        'winner':   winner,
        'rewards':  rewards,
    }
    save_results(results, RESULTS_PATH)

total_elapsed = (time.time() - t_start) / 60
print(f'\n1v1 complete.  Time: {total_elapsed:.1f} min')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 7 — 4v4 Balanced tournament
#
# Perfect combinatorial design for 16 agents:
#   5 rounds × 4 games = 20 games
#   Each agent plays exactly 5 games
#   Each pair of agents appears together exactly once
#
# If N != 16, falls back to random balanced schedule.
# ─────────────────────────────────────────────────────────
import random, time

# ── Build 4v4 schedule ────────────────────────────────────
_PERFECT_16 = [
    # Round 0
    (0,1,2,3), (4,5,6,7), (8,9,10,11), (12,13,14,15),
    # Round 1
    (0,4,8,12), (1,5,9,13), (2,6,10,14), (3,7,11,15),
    # Round 2
    (0,5,10,15), (1,4,11,14), (2,7,8,13), (3,6,9,12),
    # Round 3
    (0,6,11,13), (1,7,10,12), (2,4,9,15), (3,5,8,14),
    # Round 4
    (0,7,9,14), (1,6,8,15), (2,5,11,12), (3,4,10,13),
]

def _make_schedule(names):
    """Return list of (name_a, name_b, name_c, name_d) tuples."""
    n = len(names)
    if n == 16:
        return [tuple(names[i] for i in group) for group in _PERFECT_16]

    # Fallback: random balanced, each agent plays 5 games
    if n < 4:
        print('  [SKIP 4v4] need at least 4 agents')
        return []
    games_per_agent = 5
    schedule = []
    for round_i in range(games_per_agent):
        pool = names.copy()
        random.seed(round_i * 31337)
        random.shuffle(pool)
        for k in range(0, (n // 4) * 4, 4):
            schedule.append(tuple(pool[k:k+4]))
    return schedule


SCHEDULE_4V4 = _make_schedule(AGENT_NAMES)
results['metadata']['n_4v4_games'] = len(SCHEDULE_4V4)

done_4v4 = len(results['4v4_matches'])
remaining_4v4 = len(SCHEDULE_4V4) - done_4v4
print(f'4v4 Balanced: {len(SCHEDULE_4V4)} total games  ({done_4v4} done, {remaining_4v4} remaining)')
print(f'Each agent plays {len(SCHEDULE_4V4) * 4 // N} games.\n')

t_start = time.time()

for game_idx, group in enumerate(SCHEDULE_4V4):
    if game_idx < done_4v4:
        continue   # already played — skip

    fns   = [AGENT_FNS[name] for name in group]
    seed  = 5000 + game_idx
    label = '  '.join(group)

    t0 = time.time()
    print(f'  [{game_idx+1}/{len(SCHEDULE_4V4)}] {label} ... ', end='', flush=True)

    rewards = run_game(fns, seed=seed)
    winner  = _determine_winner(rewards, list(group))
    elapsed = time.time() - t0

    # Rank by reward (rank 1 = highest reward)
    if rewards:
        sorted_idx = sorted(range(4), key=lambda k: -(rewards[k] or 0))
        ranks = [0] * 4
        for rank, idx in enumerate(sorted_idx, 1):
            ranks[idx] = rank
    else:
        ranks = [0, 0, 0, 0]

    print(f'winner={winner}  ({elapsed:.0f}s)')

    results['4v4_matches'].append({
        'game_idx': game_idx,
        'players':  list(group),
        'winner':   winner,
        'rewards':  rewards,
        'ranks':    {group[k]: ranks[k] for k in range(4)},
    })
    save_results(results, RESULTS_PATH)

total_elapsed = (time.time() - t_start) / 60
print(f'\n4v4 complete.  Time: {total_elapsed:.1f} min')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 8 — Compute stats + rankings
# ─────────────────────────────────────────────────────────

# ── 1v1 stats ────────────────────────────────────────────
stats_1v1 = {name: {'wins': 0, 'losses': 0, 'draws': 0, 'errors': 0,
                     'games': 0, 'head_to_head': {}}
             for name in AGENT_NAMES}

for match in results['1v1_matches'].values():
    a, b, winner = match['agent_a'], match['agent_b'], match['winner']

    for name in [a, b]:
        stats_1v1[name]['games'] += 1

    if winner == 'draw':
        stats_1v1[a]['draws'] += 1
        stats_1v1[b]['draws'] += 1
        stats_1v1[a]['head_to_head'][b] = 'draw'
        stats_1v1[b]['head_to_head'][a] = 'draw'
    elif winner == 'error':
        stats_1v1[a]['errors'] += 1
        stats_1v1[b]['errors'] += 1
    elif winner == a:
        stats_1v1[a]['wins']   += 1
        stats_1v1[b]['losses'] += 1
        stats_1v1[a]['head_to_head'][b] = 'win'
        stats_1v1[b]['head_to_head'][a] = 'loss'
    elif winner == b:
        stats_1v1[b]['wins']   += 1
        stats_1v1[a]['losses'] += 1
        stats_1v1[b]['head_to_head'][a] = 'win'
        stats_1v1[a]['head_to_head'][b] = 'loss'

for name, s in stats_1v1.items():
    s['win_rate'] = round(s['wins'] / max(1, s['games']), 4)

# ── 4v4 stats ────────────────────────────────────────────
stats_4v4 = {name: {'wins': 0, 'games': 0, 'rank_sum': 0}
             for name in AGENT_NAMES}

for match in results['4v4_matches']:
    winner = match['winner']
    for name in match['players']:
        stats_4v4[name]['games'] += 1
        rank = match['ranks'].get(name, 4)
        stats_4v4[name]['rank_sum'] += rank
        if name == winner:
            stats_4v4[name]['wins'] += 1

for name, s in stats_4v4.items():
    g = max(1, s['games'])
    s['win_rate']  = round(s['wins'] / g, 4)
    s['avg_rank']  = round(s['rank_sum'] / g, 2)

# ── Rankings ──────────────────────────────────────────────
ranking_1v1 = sorted(AGENT_NAMES,
    key=lambda n: (-stats_1v1[n]['wins'], stats_1v1[n]['losses']))

ranking_4v4 = sorted(AGENT_NAMES,
    key=lambda n: (-stats_4v4[n]['wins'], stats_4v4[n]['avg_rank']))

results['1v1_stats']    = stats_1v1
results['4v4_stats']    = stats_4v4
results['ranking_1v1']  = ranking_1v1
results['ranking_4v4']  = ranking_4v4

# ── Combined tier list (avg of both rankings) ─────────────
rank1_pos = {name: i for i, name in enumerate(ranking_1v1)}
rank4_pos = {name: i for i, name in enumerate(ranking_4v4)}
combined  = sorted(AGENT_NAMES, key=lambda n: rank1_pos[n] + rank4_pos[n])
results['ranking_combined'] = combined

# Assign tiers: top 4 = hard, middle 8 = medium, bottom 4 = easy
n = len(combined)
results['tiers'] = {
    'hard':   combined[:n//4],
    'medium': combined[n//4 : 3*n//4],
    'easy':   combined[3*n//4:],
}

save_results(results, RESULTS_PATH)
print(f'Results saved to {RESULTS_PATH}')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 9 — Print leaderboard
# ─────────────────────────────────────────────────────────

print('=' * 60)
print('  1v1 LEADERBOARD  (round-robin, each agent plays', N-1, 'games)')
print('=' * 60)
print(f'  {"Rank":<5} {"Agent":<35} {"W":>4} {"L":>4} {"D":>4} {"WR%":>6}')
print(f'  {"-"*5} {"-"*35} {"-"*4} {"-"*4} {"-"*4} {"-"*6}')
for rank, name in enumerate(ranking_1v1, 1):
    s = stats_1v1[name]
    print(f'  {rank:<5} {name:<35} {s["wins"]:>4} {s["losses"]:>4} '
          f'{s["draws"]:>4} {s["win_rate"]*100:>5.1f}%')

print()
print('=' * 60)
print('  4v4 LEADERBOARD  (each agent plays', len(SCHEDULE_4V4)*4//N, 'games)')
print('=' * 60)
print(f'  {"Rank":<5} {"Agent":<35} {"Wins":>5} {"WR%":>6} {"AvgRank":>8}')
print(f'  {"-"*5} {"-"*35} {"-"*5} {"-"*6} {"-"*8}')
for rank, name in enumerate(ranking_4v4, 1):
    s = stats_4v4[name]
    print(f'  {rank:<5} {name:<35} {s["wins"]:>5} {s["win_rate"]*100:>5.1f}% {s["avg_rank"]:>8.2f}')

print()
print('=' * 60)
print('  COMBINED TIERS  (use for PPO curriculum)')
print('=' * 60)
for tier, members in results['tiers'].items():
    print(f'  {tier.upper():<8}: {chr(10).join("    " + m for m in members)}')
    print()

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 10 — Plot leaderboard chart
# ─────────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, max(6, N * 0.45)))
fig.suptitle('Enemy Agent Tournament Results', fontsize=13)

# ── 1v1 bar chart ─────────────────────────────────────────
ax   = axes[0]
names_sorted = ranking_1v1[::-1]   # bottom to top
wins   = [stats_1v1[n]['wins']   for n in names_sorted]
losses = [stats_1v1[n]['losses'] for n in names_sorted]
draws  = [stats_1v1[n]['draws']  for n in names_sorted]

y = np.arange(len(names_sorted))
ax.barh(y, wins,   color='steelblue', label='wins')
ax.barh(y, draws,  left=wins, color='gray', alpha=0.5, label='draws')
ax.barh(y, losses, left=[w+d for w,d in zip(wins,draws)],
        color='crimson', alpha=0.6, label='losses')
ax.set_yticks(y)
ax.set_yticklabels(names_sorted, fontsize=8)
ax.set_xlabel('Games')
ax.set_title('1v1 Round-Robin')
ax.axvline(N//2, color='k', linestyle='--', alpha=0.3, label=f'50% ({N//2} wins)')
ax.legend(fontsize=8)

# ── 4v4 bar chart ─────────────────────────────────────────
ax   = axes[1]
names_sorted4 = ranking_4v4[::-1]
wins4  = [stats_4v4[n]['wins']     for n in names_sorted4]
ranks4 = [stats_4v4[n]['avg_rank'] for n in names_sorted4]

color_map = plt.cm.RdYlGn_r
colors = [color_map((r - 1) / 3) for r in ranks4]
ax.barh(y, wins4, color=colors)
ax.set_yticks(y)
ax.set_yticklabels(names_sorted4, fontsize=8)
ax.set_xlabel('Wins  (colour = avg rank: green=1st, red=4th)')
ax.set_title('4v4 Balanced')

plt.tight_layout()
CHART_PATH = WORK / 'tournament_chart.png'
plt.savefig(CHART_PATH, dpi=120, bbox_inches='tight')
plt.show()
print(f'Chart saved: {CHART_PATH}')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 11 — Output files + close log
# ─────────────────────────────────────────────────────────
print('Output files (download from Output tab):')
for name in ['tournament_results.json', 'tournament_chart.png', 'tournament.log']:
    f = WORK / name
    if f.exists():
        print(f'  {name:<35}  {f.stat().st_size/1e3:.1f} KB')

print()
print('Place tournament_results.json in rl/ to use for opponent curriculum.')
print('The tiers field gives you easy/medium/hard opponent groupings directly.')

try:
    import time as _t
    print(f'\n# Log ended : {_t.strftime("%Y-%m-%d %H:%M:%S UTC", _t.gmtime())}')
    _tee.flush()
    _tee.close()
except Exception as e:
    print(f'Log close: {e}')